# Study 2: Subgroup Analysis

Identify where models perform well vs poorly:
- **Age groups**: 18-30, 31-40, 41-50, 51-55
- **Sex**: Male, Female
- **Comorbidities** (if available): ADHD, Anxiety, Depression

Uses the best model from Study 1 (XGBoost or LightGBM) per cohort. For each subgroup: 5-fold CV and metrics. Output: subgroup_comparison_table.csv and optional forest plot.

In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd

_cwd = os.path.abspath(os.getcwd())
REPO_ROOT = os.path.dirname(_cwd) if os.path.basename(_cwd) == 'notebooks' else _cwd
if not os.path.isdir(os.path.join(REPO_ROOT, 'data')):
    REPO_ROOT = os.path.dirname(REPO_ROOT)
sys.path.insert(0, os.path.join(REPO_ROOT, 'src'))

from study_utils import (
    load_cohort_c4,
    load_cohort_card,
    load_cohort_ybt,
    train_with_cv,
    evaluate_model,
    bootstrap_ci_auroc,
    get_models,
    RESULTS_DIR,
    AGE_GROUPS_STUDY,
    RANDOM_STATE,
    COMOBIDITY_COLUMNS,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score

STUDY1_DIR = os.path.join(RESULTS_DIR, 'study1_within_cohort')
STUDY2_DIR = os.path.join(RESULTS_DIR, 'study2_subgroups')
for sub in ['age_stratified', 'sex_stratified', 'comorbidity_stratified']:
    os.makedirs(os.path.join(STUDY2_DIR, sub), exist_ok=True)

C4_PATH = os.path.join(REPO_ROOT, 'data', 'processed', 'data_c4_final_recreated_cleaned.csv')
CARD_ALIGNED_PATH = os.path.join(REPO_ROOT, 'data', 'processed', 'card_aligned.csv')
YBT_ALIGNED_PATH = os.path.join(REPO_ROOT, 'data', 'processed', 'ybt_aligned.csv')
_default_ybt = os.path.expanduser('~/Library/CloudStorage/OneDrive-UniversityofCambridge/Documents/PhD/data/YBT.csv')
_repo_ybt = os.path.join(REPO_ROOT, 'data', 'raw', 'YBT.csv')
YBT_RAW_PATH = os.environ.get('YBT_PATH', _default_ybt if os.path.isfile(_default_ybt) else _repo_ybt)

## 1. Load cohorts (same as Study 1, no balance to keep subgroup sizes)

In [ ]:
df_c4, feat_c4, target_c4 = load_cohort_c4(C4_PATH, age_min=18, age_max=55, balance_50_50=False, apply_aq_filter=True)
df_card = None
if os.path.isfile(CARD_ALIGNED_PATH):
    df_card, feat_card, target_card = load_cohort_card(CARD_ALIGNED_PATH, age_min=18, age_max=55, balance_50_50=False, apply_aq_filter=True)
else:
    df_card, feat_card, target_card = None, None, None
ybt_path = YBT_ALIGNED_PATH if os.path.isfile(YBT_ALIGNED_PATH) else YBT_RAW_PATH
df_ybt, feat_ybt, target_ybt = load_cohort_ybt(ybt_path, age_min=18, age_max=55, balance_50_50=False, apply_aq_filter=True)

def add_study_age_groups(df, age_col='age'):
    df = df.copy()
    def ag(age):
        if age <= 30: return '18-30'
        if age <= 40: return '31-40'
        if age <= 50: return '41-50'
        return '51-55'
    df['age_strata'] = df[age_col].apply(ag)
    return df

df_c4 = add_study_age_groups(df_c4)
if df_card is not None:
    df_card = add_study_age_groups(df_card)
df_ybt = add_study_age_groups(df_ybt)

## 2. Subgroup evaluation helper

For each (cohort, subgroup_type, subgroup_value): filter data, 5-fold CV with XGBoost, compute AUROC and 95% CI.

In [ ]:
def evaluate_subgroup(X, y, feature_names, n_splits=5):
    if len(np.unique(y)) < 2 or len(y) < 50:
        return {'n': len(y), 'auroc': np.nan, 'ci_lower': np.nan, 'ci_upper': np.nan, 'sensitivity': np.nan, 'specificity': np.nan, 'f1': np.nan}
    from sklearn.preprocessing import StandardScaler
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    scaler = StandardScaler()
    Xs = scaler.fit_transform(X)
    model = get_models()['xgboost']
    probas = np.zeros_like(y, dtype=float)
    for train_idx, val_idx in skf.split(Xs, y):
        m = get_models()['xgboost']
        m.fit(Xs[train_idx], y[train_idx])
        probas[val_idx] = m.predict_proba(Xs[val_idx])[:, 1]
    auroc = roc_auc_score(y, probas)
    ci_lo, ci_hi = bootstrap_ci_auroc(y, probas)
    pred = (probas >= 0.5).astype(int)
    tp = ((y==1)&(pred==1)).sum()
    fn = ((y==1)&(pred==0)).sum()
    tn = ((y==0)&(pred==0)).sum()
    fp = ((y==0)&(pred==1)).sum()
    sens = tp/(tp+fn) if (tp+fn)>0 else np.nan
    spec = tn/(tn+fp) if (tn+fp)>0 else np.nan
    from sklearn.metrics import f1_score
    f1 = f1_score(y, pred, zero_division=0)
    return {'n': len(y), 'auroc': auroc, 'ci_lower': ci_lo, 'ci_upper': ci_hi, 'sensitivity': sens, 'specificity': spec, 'f1': f1}

## 3. Age-stratified results

In [ ]:
age_results = []
cohorts = [('C4', df_c4, feat_c4, target_c4), ('Dataset3', df_ybt, feat_ybt, target_ybt)]
if df_card is not None:
    cohorts.insert(1, ('CARD', df_card, feat_card, target_card))
for cohort_name, df, feat, target in cohorts:
    if df is None: continue
    for ag in ['18-30', '31-40', '41-50', '51-55']:
        sub = df[df['age_strata'] == ag]
        if len(sub) < 30: continue
        X = sub[feat].fillna(0).values
        y = sub[target].values
        r = evaluate_subgroup(X, y, feat)
        r['Cohort'] = cohort_name
        r['Subgroup'] = 'Age'
        r['Category'] = ag
        age_results.append(r)

if age_results:
    age_df = pd.DataFrame(age_results)
    age_df.to_csv(os.path.join(STUDY2_DIR, 'age_stratified', 'subgroup_comparison_table.csv'), index=False)
    print(age_df.to_string(index=False))

## 4. Sex-stratified results

Requires a consistent sex column (e.g. Male/Female or 0/1). Map sex_num or sex to a binary if needed.

In [ ]:
sex_results = []
cohorts = [('C4', df_c4, feat_c4, target_c4), ('Dataset3', df_ybt, feat_ybt, target_ybt)]
if df_card is not None:
    cohorts.insert(1, ('CARD', df_card, feat_card, target_card))
for cohort_name, df, feat, target in cohorts:
    if df is None: continue
    if 'sex_num' not in df.columns and 'sex' not in df.columns: continue
    sex_col = 'sex_num' if 'sex_num' in df.columns else 'sex'
    for sex_val, label in [(0, 'Male'), (1, 'Female')]:
        sub = df[df[sex_col] == sex_val]
        if len(sub) < 30: continue
        X = sub[feat].fillna(0).values
        y = sub[target].values
        r = evaluate_subgroup(X, y, feat)
        r['Cohort'] = cohort_name
        r['Subgroup'] = 'Sex'
        r['Category'] = label
        sex_results.append(r)

if sex_results:
    sex_df = pd.DataFrame(sex_results)
    sex_df.to_csv(os.path.join(STUDY2_DIR, 'sex_stratified', 'subgroup_comparison_table.csv'), index=False)
    print(sex_df.to_string(index=False))

## 5. Comorbidity-stratified results

Requires has_adhd, has_anxiety, has_depression in the cohort data (from raw or added by pipelines). C4: re-run data_pipeline_recreation to get these. YBT: parsed from raw diagnosis text. CARD: placeholder 0s until comorbidity data available.

In [ ]:
comorbidity_results = []
cohorts = [('C4', df_c4, feat_c4, target_c4), ('Dataset3', df_ybt, feat_ybt, target_ybt)]
if df_card is not None:
    cohorts.insert(1, ('CARD', df_card, feat_card, target_card))
for cohort_name, df, feat, target in cohorts:
    if df is None: continue
    present = [c for c in COMOBIDITY_COLUMNS if c in df.columns]
    if not present: continue
    for col in present:
        label = col.replace('has_', '').capitalize()
        sub = df[df[col] == 1]
        if len(sub) < 30: continue
        X = sub[feat].fillna(0).values
        y = sub[target].values
        r = evaluate_subgroup(X, y, feat)
        r['Cohort'] = cohort_name
        r['Subgroup'] = 'Comorbidity'
        r['Category'] = label
        comorbidity_results.append(r)

if comorbidity_results:
    comorb_df = pd.DataFrame(comorbidity_results)
    comorb_df.to_csv(os.path.join(STUDY2_DIR, 'comorbidity_stratified', 'subgroup_comparison_table.csv'), index=False)
    print(comorb_df.to_string(index=False))
else:
    print('No comorbidity columns (has_adhd, has_anxiety, has_depression) in cohort data. Re-run data_pipeline_recreation for C4; YBT raw has them from diagnosis text.')

## 6. Aggregate subgroup table and forest plot

Combine age, sex, and comorbidity into one table; optional matplotlib forest plot.

In [ ]:
all_sub = []
for path in [os.path.join(STUDY2_DIR, 'age_stratified', 'subgroup_comparison_table.csv'),
               os.path.join(STUDY2_DIR, 'sex_stratified', 'subgroup_comparison_table.csv'),
               os.path.join(STUDY2_DIR, 'comorbidity_stratified', 'subgroup_comparison_table.csv')]:
    if os.path.isfile(path):
        all_sub.append(pd.read_csv(path))
if all_sub:
    combined = pd.concat(all_sub, ignore_index=True)
    combined.to_csv(os.path.join(STUDY2_DIR, 'subgroup_comparison_table.csv'), index=False)
    print(combined.to_string(index=False))

import matplotlib.pyplot as plt
if all_sub and 'auroc' in combined.columns and 'ci_lower' in combined.columns:
    fig, ax = plt.subplots(figsize=(8, 6))
    combined_valid = combined.dropna(subset=['auroc', 'ci_lower', 'ci_upper'])
    y_pos = range(len(combined_valid))
    ax.errorbar(combined_valid['auroc'], y_pos, xerr=[combined_valid['auroc']-combined_valid['ci_lower'], combined_valid['ci_upper']-combined_valid['auroc']], fmt='o', capsize=3)
    ax.set_yticks(y_pos)
    ax.set_yticklabels([f"{r['Cohort']} {r['Category']} (n={int(r['n'])})" for _, r in combined_valid.iterrows()])
    ax.axvline(0.5, color='gray', linestyle='--')
    ax.set_xlabel('AUROC')
    ax.set_title('Subgroup AUROC (95% CI)')
    plt.tight_layout()
    plt.savefig(os.path.join(STUDY2_DIR, 'subgroup_forest_plot.png'), dpi=150, bbox_inches='tight')
    plt.show()